In [7]:
import re
import time
from bs4 import BeautifulSoup
import requests
from xml.etree import ElementTree as ET
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from webdriver_manager.microsoft import EdgeChromiumDriverManager

In [19]:
API_BASE = "https://api.poiskkino.dev"
API_KEY = "GST9C9T-VE6M90G-JG8J2PF-PSB093D"
ENDPOINT = "/v1.4/movie"

headers_api = {
    "X-API-KEY": API_KEY,
    "Accept": "application/json"
}

SELECT_FIELDS = [
    "name",
    "year",
    "shortDescription",
    "description",
    "genres",
    "countries"
]

base_params = {
    "limit": 250,
    "selectFields": ",".join(SELECT_FIELDS),
    "sortField": "rating.kp",
    "sortType": "-1",
    "notNullFields": "rating.kp"
}

def fetch_movies_page(page=1):
    params_page = base_params.copy()
    params_page["page"] = page
    try:
        resp = requests.get(API_BASE + ENDPOINT, headers=headers_api, params=params_page, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"Error on page {page}: {e}")
        return {}

def collect_movies_api(target=250, max_pages=20):
    result = []
    seen = set()
    page = 1

    while len(result) < target and page <= max_pages:
        print(f"  API: Fetching page {page}...")
        data = fetch_movies_page(page)
        if not data:
            break

        movies = data.get("docs", [])
        if not movies:
            print(f"  API: No more movies on page {page}.")
            break

        for movie in movies:
            title = movie.get("name", "").strip()
            year = str(movie.get("year", "")).strip() if movie.get("year") else ""
            description = (movie.get("shortDescription") or movie.get("description", "")).strip()

            if not title or not year or not description:
                continue

            key = (title.lower(), year)
            if key in seen:
                continue
            seen.add(key)

            genres_list = movie.get("genres", [])
            genres = ", ".join([g.get("name", "") for g in genres_list if g.get("name")])
            countries_list = movie.get("countries", [])
            countries = ", ".join([c.get("name", "") for c in countries_list if c.get("name")])

            result.append({
                "title": title,
                "year": year,
                "country": countries,
                "genres": genres,
                "description": description
            })

            if len(result) >= target:
                break

        page += 1
        time.sleep(0.5)

    print(f"API: collected {len(result)} movies.")
    return result[:target]

print("\n--- Poiskkino API ---")
movies = collect_movies_api(target=250)

for movie in movies:
    if not movie['country']:
        movie['country'] = 'Unknown'
    if not movie['genres']:
        movie['genres'] = 'Unknown'

df = pd.DataFrame(movies)
df.to_csv('200_api.csv', index=False, encoding='utf-8-sig')
print(f"\nSaved {len(movies)} movies to 'movies_api.csv'")


--- Poiskkino API ---
  API: Fetching page 1...
  API: Fetching page 2...
API: collected 250 movies.

Saved 250 movies to 'movies_api.csv'


In [16]:
!pip install requests beautifulsoup4 lxml pandas

Defaulting to user installation because normal site-packages is not writeable


In [25]:
import requests, time, re, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

HEADERS = {'User-Agent': 'Mozilla/5.0'}

def get_soup(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        return BeautifulSoup(r.text, 'html.parser')
    except:
        return None

def get_year(soup):
    # Try infobox
    ib = soup.find('table', class_='infobox')
    if ib:
        for row in ib.find_all('tr'):
            th = row.find('th')
            if th and any(x in th.get_text(strip=True).lower() for x in ['released', 'year']):
                td = row.find('td')
                if td:
                    m = re.search(r'\b(19|20)\d{2}\b', td.text)
                    if m: return m.group()
    # Try first paragraph
    div = soup.find('div', class_='mw-parser-output')
    if div:
        p = div.find('p', recursive=False)
        if p:
            m = re.search(r'\b(19|20)\d{2}\b', p.text)
            if m: return m.group()
    # Try anywhere
    body = soup.get_text()
    m = re.search(r'\b(19|20)\d{2}\b', body)
    if m: return m.group()
    return 'Unknown'

def get_description(soup):
    div = soup.find('div', class_='mw-parser-output')
    if not div:
        return 'No description available.'
    # Get first paragraph that has text
    for p in div.find_all('p', recursive=False):
        text = p.get_text(strip=True)
        if text:
            return text[:500]
    # Fallback: any paragraph
    for p in div.find_all('p'):
        text = p.get_text(strip=True)
        if text:
            return text[:500]
    return 'No description available.'

def get_country(soup):
    ib = soup.find('table', class_='infobox')
    if ib:
        for row in ib.find_all('tr'):
            th = row.find('th')
            if th and 'country' in th.get_text(strip=True).lower():
                td = row.find('td')
                if td:
                    return td.get_text(strip=True).split('\n')[0][:100]
    return 'Unknown'

def get_movie(url):
    soup = get_soup(url)
    if not soup: return None
    title = soup.find('h1')
    if not title: return None
    title = title.text.strip()
    desc = get_description(soup)
    if not desc:  # must have something
        desc = 'No description available.'
    year = get_year(soup)
    country = get_country(soup)
    return {'title': title, 'year': year, 'country': country, 'description': desc}

def extract_links_from_table(url, col, name):
    print(f"\n--- {name} ---")
    soup = get_soup(url)
    if not soup: return []
    table = soup.find('table', class_='wikitable')
    if not table:
        print("  No wikitable")
        return []
    links = []
    for row in table.find_all('tr')[1:]:
        cells = row.find_all('td')
        if len(cells) > col:
            a = cells[col].find('a')
            if a and a.get('href') and a['href'].startswith('/wiki/'):
                if ':' not in a['href'].split('/wiki/')[1][:5]:
                    links.append(urljoin('https://en.wikipedia.org', a['href']))
    print(f"  Found {len(links)} links")
    return links

def extract_links_from_div(url, name):
    print(f"\n--- {name} ---")
    soup = get_soup(url)
    if not soup: return []
    content = soup.find('div', class_='mw-parser-output')
    if not content: return []
    links = set()
    for a in content.find_all('a', href=True):
        href = a['href']
        if href.startswith('/wiki/') and ':' not in href:
            if not any(x in href.lower() for x in ['/list_of', 'portal', 'wikipedia', 'file:', 'help:', 'template']):
                links.add(urljoin('https://en.wikipedia.org', href))
    print(f"  Found {len(links)} links")
    return list(links)

# ========== SOURCES ==========
all_links = []

# AFI lists
all_links.extend(extract_links_from_table("https://en.wikipedia.org/wiki/AFI%27s_100_Years...100_Movies_(10th_Anniversary_Edition)", 1, "AFI 2007"))
all_links.extend(extract_links_from_table("https://en.wikipedia.org/wiki/AFI%27s_100_Years...100_Movies", 0, "AFI 1998"))

# National Film Registry (1998-2024) - very reliable source
for year in range(1998, 2025):
    url = f"https://en.wikipedia.org/wiki/National_Film_Registry_{year}"
    links = extract_links_from_table(url, 1, f"NFR {year}")
    all_links.extend(links)
    if len(all_links) > 1000: break  # enough

# BAFTA Best Film
all_links.extend(extract_links_from_table("https://en.wikipedia.org/wiki/BAFTA_Award_for_Best_Film", 2, "BAFTA Best Film"))

# Criterion Collection (has a list of films)
all_links.extend(extract_links_from_div("https://en.wikipedia.org/wiki/List_of_Criterion_Collection_releases", "Criterion Collection"))

# IMDb Top 250 (if table exists)
all_links.extend(extract_links_from_table("https://en.wikipedia.org/wiki/IMDb_Top_250_movies", 1, "IMDb Top 250"))

# List of films considered the best
all_links.extend(extract_links_from_div("https://en.wikipedia.org/wiki/List_of_films_considered_the_best", "Best Films"))

# Palme d'Or (filter out director pages)
palme = extract_links_from_table("https://en.wikipedia.org/wiki/Palme_d%27Or", 1, "Palme d'Or")
palme = [l for l in palme if not re.search(r'\d{4}-\d{4}', l) and 'director' not in l.lower()]
all_links.extend(palme)

# Golden Lion
golden = extract_links_from_table("https://en.wikipedia.org/wiki/Golden_Lion", 1, "Golden Lion")
golden = [l for l in golden if not re.search(r'\d{4}-\d{4}', l)]
all_links.extend(golden)

# Remove duplicates
all_links = list(dict.fromkeys(all_links))
print(f"\nTotal unique movie links: {len(all_links)}")

# Scrape until we have 250
movies = []
seen = set()
for i, url in enumerate(all_links):
    if len(movies) >= 250:
        break
    print(f"  {i+1}/{len(all_links)}: {url.split('/')[-1]}")
    m = get_movie(url)
    if not m:
        print("    Failed")
        continue
    key = (m['title'].lower(), m['year'])
    if key in seen:
        continue
    seen.add(key)
    movies.append(m)
    print(f"    Added: {m['title']} ({m['year']}) - desc length: {len(m['description'])}")
    time.sleep(0.2)  # faster

print(f"\nFinal count: {len(movies)} movies")

# Save
df = pd.DataFrame(movies)
df.to_csv('wikipedia_movies_250.csv', index=False, encoding='utf-8-sig')
print(f"Saved to wikipedia_movies_250.csv")

if len(movies) > 0:
    print("\nSample:")
    print(df[['title', 'year']].head(10))


--- AFI 2007 ---
  Found 99 links

--- AFI 1998 ---
  Found 122 links

--- NFR 1998 ---
  No wikitable

--- NFR 1999 ---
  No wikitable

--- NFR 2000 ---
  No wikitable

--- NFR 2001 ---
  No wikitable

--- NFR 2002 ---
  No wikitable

--- NFR 2003 ---
  No wikitable

--- NFR 2004 ---
  No wikitable

--- NFR 2005 ---
  No wikitable

--- NFR 2006 ---
  No wikitable

--- NFR 2007 ---
  No wikitable

--- NFR 2008 ---
  No wikitable

--- NFR 2009 ---
  No wikitable

--- NFR 2010 ---
  No wikitable

--- NFR 2011 ---
  No wikitable

--- NFR 2012 ---
  No wikitable

--- NFR 2013 ---
  No wikitable

--- NFR 2014 ---
  No wikitable

--- NFR 2015 ---
  No wikitable

--- NFR 2016 ---
  No wikitable

--- NFR 2017 ---
  No wikitable

--- NFR 2018 ---
  No wikitable

--- NFR 2019 ---
  No wikitable

--- NFR 2020 ---
  No wikitable

--- NFR 2021 ---
  No wikitable

--- NFR 2022 ---
  No wikitable

--- NFR 2023 ---
  No wikitable

--- NFR 2024 ---
  No wikitable

--- BAFTA Best Film ---
  Found 12 li